<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Face%20Mask%20Detection%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Real-Time Face Mask Detection System

This notebook will guide you through building a real-time face mask detection system using a webcam. The system will:

1.  **Load a pre-trained face mask detection model.**
2.  **Access the webcam feed.**
3.  **Detect faces in each frame.**
4.  **Classify if detected faces are wearing a mask or not.**
5.  **Display bounding boxes and labels (`Mask`/`No Mask`).**
6.  **Implement an alert system for 'No Mask' detection.**

Let's start by installing the necessary libraries and loading the model.

In [1]:
# Install necessary libraries. This might take a moment.
!pip install opencv-python tensorflow keras numpy scikit-image

In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
import os
from google.colab.patches import cv2_imshow # For displaying images in Colab

print("Libraries imported successfully.")

Libraries imported successfully.


### Load Face Detector and Face Mask Detector Models

We'll need two models:
1.  A **face detector** (e.g., OpenCV's Caffe model) to localize faces in the video stream.
2.  A **face mask detector** (a custom-trained deep learning model) to classify if a detected face is wearing a mask or not.

First, let's download the pre-trained models. These models are typically stored as `.prototxt` and `.caffemodel` for the face detector, and `.h5` for the mask detector.

In [10]:
# Download pre-trained face detector models (if not already present)
# The 'deploy.prototxt' defines the model architecture
# The 'res10_300x300_ssd_iter_140000.caffemodel' contains the pre-trained weights

FACE_DETECTOR_PROTOTXT = 'deploy.prototxt'
FACE_DETECTOR_MODEL = 'res10_300x300_ssd_iter_140000.caffemodel'
FACE_MASK_MODEL = 'mask_detector.h5' # Expecting a .h5 or .keras model compatible with Keras 3

# Check if files exist, if not, download them
if not os.path.exists(FACE_DETECTOR_PROTOTXT):
    print("Downloading deploy.prototxt...")
    !wget https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/dnn/deploy.prototxt -O deploy.prototxt
if not os.path.exists(FACE_DETECTOR_MODEL):
    print("Downloading res10_300x300_ssd_iter_140000.caffemodel...")
    !wget https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20180205_fd/res10_300x300_ssd_iter_140000.caffemodel -O res10_300x300_ssd_iter_140000.caffemodel

# Load the face detector model
prototxtPath = os.path.join(os.getcwd(), FACE_DETECTOR_PROTOTXT)
weightsPath = os.path.join(os.getcwd(), FACE_DETECTOR_MODEL)
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)
print(f"Face Detector model loaded successfully.")

# Load the face mask detector model
# IMPORTANT: The previously downloaded 'mask_detector.h5' was incompatible with Keras 3.
# You need to provide your own trained model here.
# It MUST be in a format compatible with Keras 3 (.h5 or .keras).
# Upload your model to your Colab environment or provide a direct download link.
# If you need to train one, ensure it's saved with Keras 3 or a compatible TensorFlow version.
if not os.path.exists(FACE_MASK_MODEL):
    print(f"\nWARNING: Face mask detection model '{FACE_MASK_MODEL}' not found.")
    print("Please upload your trained model (in .h5 or .keras format) to your Colab environment,")
    print("or provide a valid download URL for a Keras 3 compatible model.")
    print("Example: !wget https://example.com/your_model.h5 -O mask_detector.h5")
else:
    print(f"\nAttempting to load face mask detection model: {FACE_MASK_MODEL}")

try:
    maskNet = load_model(FACE_MASK_MODEL)
    print(f"Models loaded successfully: Face Detector and Face Mask Detector ({FACE_MASK_MODEL})")
except Exception as e:
    print(f"Error loading mask detector model '{FACE_MASK_MODEL}': {e}")
    print("This error often occurs due to Keras version incompatibility.")
    print("Please ensure your 'mask_detector.h5' or '.keras' model was saved using Keras 3 (or a compatible TensorFlow 2.x environment).")
    print("You may need to train a new model or convert an existing one to be compatible with Keras 3.")

Face Detector model loaded successfully.

Attempting to load face mask detection model: mask_detector.h5
Error loading mask detector model 'mask_detector.h5': Error when deserializing class 'Conv2D' using config={'name': 'Conv1', 'trainable': False, 'dtype': 'float32', 'filters': 32, 'kernel_size': [3, 3], 'strides': [2, 2], 'padding': 'valid', 'data_format': 'channels_last', 'dilation_rate': [1, 1], 'activation': 'linear', 'use_bias': False, 'kernel_initializer': {'class_name': 'GlorotUniform', 'config': {'seed': None, 'dtype': 'float32'}}, 'bias_initializer': {'class_name': 'Zeros', 'config': {'dtype': 'float32'}}, 'kernel_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None}.

Exception encountered: <class 'keras.src.initializers.random_initializers.GlorotUniform'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()